In [ ]:
import os
import asyncio
from telethon import TelegramClient, events
from dotenv import load_dotenv
from signal_parser import SignalParser 

load_dotenv()

class TelegramHandler:
    def __init__(self, channels: list):
        self.api_id = int(os.getenv("TELEGRAM_API_ID"))
        self.api_hash = os.getenv("TELEGRAM_API_HASH")
        self.session_name = "signals_session"
        self.channels = channels

        # Initialize the Telethon client
        self.client = TelegramClient(self.session_name, self.api_id, self.api_hash)

        # Register the event handler
        self.client.add_event_handler(self.signal_handler, events.NewMessage(chats=self.channels))

    async def signal_handler(self, event):
        """
        Called on every new message from the subscribed channels.
        """
        raw_text = event.raw_text
        print("\n📩 New message:")
        print(raw_text)

        # Parse the signal using your SignalParser
        parsed = SignalParser.parse(raw_text)

        if parsed:
            print("\n📌 Parsed Signal:")
            print(parsed)
            # TODO: send parsed signal to trade executor
        else:
            print("❌ Message ignored (not a signal alert)")

    async def start(self):
        """
        Start the client and listen for messages indefinitely.
        """
        await self.client.start()
        print("✅ Telegram client started. Listening for signals...")
        await self.client.run_until_disconnected()



In [1]:
from signal_parser import SignalParser 

text = """🔔SIGNAL ALERT🔔 
NZDUSD: SELL 
TP 1: 0.56681 
TP 2: 0.56535 
TP 3: 0.56163 
USE LOW LOT 
SELL NOW 
Sl : 0.56991"""
parser = SignalParser()
text = parser.parse(text)
text

{'symbol': 'NZDUSD',
 'action': 'SELL',
 'entry_type': 'MARKET_EXECUTION',
 'stop_loss': 0.56991,
 'take_profit_1': 0.56681,
 'take_profit_2': 0.56535,
 'take_profit_3': 0.56163,
 'raw_tps': [0.56681, 0.56535, 0.56163],
 'notes': 'Low Lot'}

In [2]:
import MetaTrader5 as mt5

# Initialize connection
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    # You may need to specify the path to your MT5 terminal's executable
    # Example: mt5.initialize(path="C:\\Program Files\\MetaTrader 5\\terminal64.exe")
    quit()

# Connection successful - request data
print("✅ MT5 initialized successfully.")

# Get account info (optional, but good practice)
account_info = mt5.account_info()
if account_info:
    print(f"Account Login: {account_info.login}, Balance: {account_info.balance}")
else:
    print("Failed to retrieve account information.")

# Close connection when done
mt5.shutdown()

✅ MT5 initialized successfully.
Account Login: 5043442344, Balance: 100000.0


True

In [ ]:
import MetaTrader5 as mt5
import time

# 1. Initialize Connection
if not mt5.initialize():
    print("❌ initialize() failed, error code =", mt5.last_error())
    quit()
    
# 2. Select Symbol
symbol = "EURUSD"
if not mt5.symbol_select(symbol, True):
    print(f"❌ Failed to select {symbol}")
    mt5.shutdown()
    quit()

# 3. Get Price and Specs
symbol_info = mt5.symbol_info(symbol)
if symbol_info is None:
    print(f"❌ Failed to get symbol info for {symbol}")
    mt5.shutdown()
    quit()

# Get the last tick data to find the current Ask price
tick = mt5.symbol_info_tick(symbol)
# Use the Ask price for a BUY order
price = tick.ask 

# Define other parameters
lot = 0.01  # Trading volume (must be a float!)
deviation = 20 # Maximum acceptable deviation from the price, in points
magic = 10000 # Unique ID for your bot/script

f


In [1]:
import MetaTrader5 as mt

mt.initialize()

ticker = 'EURUSD'
qty = 0.01
buy_order = mt.ORDER_TYPE_BUY
sell_order = mt.ORDER_TYPE_SELL
buy_price =  mt.symbol_info_tick('EURUSD').ask
sell_price = mt.symbol_info_tick('EURUSD').bid
sl_pct = 0.05
tp_pct = 0.1
buy_sl = buy_price * (1-sl_pct)
buy_tp = buy_price * (1+tp_pct)
sell_sl = sell_price * (1+sl_pct)
sell_tp = sell_price *(1-tp_pct)

def create_orders (ticker, qty, order_type, price, sl, tp):
    request = {
        'action': mt.TRADE_ACTION_DEAL,
        'symbol': ticker,
        'price': price,
        'sl': sl,
        'tp': tp,
        'type': order_type,
        'volume' : qty,
        'type_time': mt.ORDER_TIME_GTC,
        'type_filling': mt.ORDER_FILLING_IOC,
        'comment': 'python open position'  
    }
    
    result = mt.order_send(request)
    print(result)

In [18]:

# --- ASSUMED SETUP ---
# You must run this code AFTER initialization and getting price/symbol info
symbol = "EURUSD"
lot = 0.10
magic = 10000 
price = mt.symbol_info_tick('EURUSD').ask # Current Ask price fetched from mt.symbol_info_tick(symbol)
# ---------------------

market_buy_request = {
    # 1. Action & Type
    "action": mt.TRADE_ACTION_DEAL,    # Immediate execution
    "type": mt.ORDER_TYPE_BUY,         # Buy direction
    
    # 2. Instrument & Volume
    "symbol": symbol,
    "volume": lot,
    
    # 3. Price & Slippage
    "price": price,                     # The price you expect to enter at
    "deviation": 20,                    # Max slippage (in points)
    
    # 4. Safety & Comment
    "magic": magic,
    "comment": "Market Buy via Python",
    
    # 5. Order Lifetime
    "type_time": mt.ORDER_TIME_GTC,    # Good 'til Cancelled
    "type_filling": mt.ORDER_FILLING_FOK, # Immediate or Cancel
}
mt.order_send(market_buy_request)


OrderSendResult(retcode=10009, deal=54094127335, order=54298182892, volume=0.1, price=1.17064, bid=0.0, ask=0.0, comment='Request executed', request_id=2806699683, retcode_external=0, request=TradeRequest(action=1, magic=10000, order=0, symbol='EURUSD', volume=0.1, price=1.17064, stoplimit=0.0, sl=0.0, tp=0.0, deviation=20, type=0, type_filling=0, type_time=0, expiration=0, comment='Market Buy via Python', position=0, position_by=0))

In [3]:
def ensure_symbol(symbol):
        """Ensure the symbol is available and selected in MT5."""
        try:
            if not mt.symbol_select(symbol, True):
                print(f"❌ Could not select symbol: {symbol}")
                return False
            return True
        except Exception as e:
            print(f"❌ Error selecting symbol {symbol}: {e}")
            return False
# ensure_symbol('EURUSD')

def get_price(symbol):
            """Helper to get current Ask/Bid prices for a symbol."""
            try:
                if not ensure_symbol(symbol):
                    return None, None
                    
                tick = mt.symbol_info_tick(symbol)
                print(tick)
                if tick is None or tick.last == 0.0:
                    print(f"❌ Failed to get valid tick data for {symbol}.")
                    return None, None
                    
                return tick.ask, tick.bid
            except Exception as e:
                print(f"❌ Exception retrieving price for {symbol}: {e}")
                return None, None
get_price('EURUSD')

Tick(time=1765472648, bid=1.17356, ask=1.17357, last=0.0, volume=0, time_msc=1765472648899, flags=1030, volume_real=0.0)
❌ Failed to get valid tick data for EURUSD.


(None, None)

In [8]:
create_orders(ticker,qty,order_type=buy_order,price=buy_price,sl=buy_sl,tp=buy_tp)

OrderSendResult(retcode=10030, deal=0, order=0, volume=0.0, price=0.0, bid=0.0, ask=0.0, comment='Unsupported filling mode', request_id=0, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='EURUSD', volume=0.01, price=1.16346, stoplimit=0.0, sl=1.105287, tp=1.279806, deviation=0, type=0, type_filling=1, type_time=0, expiration=0, comment='python open position', position=0, position_by=0))


In [2]:
import MetaTrader5 as mt5

if not mt5.initialize():
    print("initialize() failed, error code =",mt5.last_error())
    quit()
 
# attempt to enable the display of the EURJPY symbol in MarketWatch
selected=mt5.symbol_select("EURJPY",True)
if not selected:
    print("Failed to select EURJPY")
    mt5.shutdown()
    quit()
 
# display EURJPY symbol properties
symbol_info=mt5.symbol_info("BTCUSD")
if symbol_info!=None:
    # display the terminal data 'as is'    
    print(symbol_info)
    print("BTCUSD: spread =",symbol_info.spread,"  digits =",symbol_info.digits)
    # display symbol properties as a list
    print("Show symbol_info(\"BTCUSD\")._asdict():")
    symbol_info_dict = mt5.symbol_info("BTCUSD")._asdict()
    for prop in symbol_info_dict:
        print("  {}={}".format(prop, symbol_info_dict[prop]))
 
# shut down connection to the MetaTrader 5 terminal

mt5.symbol_info_tick('BTCUSD')
mt5.shutdown()
 

True

In [15]:
# display data on the MetaTrader 5 package
print("MetaTrader5 package author: ", mt5.__author__)
print("MetaTrader5 package version: ", mt5.__version__)
 
# establish connection to the MetaTrader 5 terminal
if not mt5.initialize():
    print("initialize() failed, error code =",mt5.last_error())
    quit()
 
# prepare the buy request structure
symbol = "USDJPY"
symbol_info = mt5.symbol_info(symbol)
if symbol_info is None:
    print(symbol, "not found, can not call order_check()")
    mt5.shutdown()
    quit()
 
# if the symbol is unavailable in MarketWatch, add it
if not symbol_info.visible:
    print(symbol, "is not visible, trying to switch on")
    if not mt5.symbol_select(symbol,True):
        print("symbol_select({}}) failed, exit",symbol)
        mt5.shutdown()
        quit()
 
lot = 0.1
point = mt5.symbol_info(symbol).point
price = mt5.symbol_info_tick(symbol).ask
deviation = 20
request = {
    "action": mt5.TRADE_ACTION_DEAL,
    "symbol": symbol,
    "volume": lot,
    "type": mt5.ORDER_TYPE_BUY,
    "price": price,
    "sl": price - 100 * point,
    "tp": price + 100 * point,
    "deviation": deviation,
    "magic": 234000,
    "comment": "python script open",
    "type_time": mt5.ORDER_TIME_GTC,
    "type_filling": mt5.ORDER_FILLING_RETURN,
}
 
# send a trading request
result = mt5.order_send(request)
# check the execution result
print("1. order_send(): by {} {} lots at {} with deviation={} points".format(symbol,lot,price,deviation));
if result.retcode != mt5.TRADE_RETCODE_DONE:
    print("2. order_send failed, retcode={}".format(result.retcode))
    # request the result as a dictionary and display it element by element
    result_dict=result._asdict()
    for field in result_dict.keys():
        print("   {}={}".format(field,result_dict[field]))
        # if this is a trading request structure, display it element by element as well
        if field=="request":
            traderequest_dict=result_dict[field]._asdict()
            for tradereq_filed in traderequest_dict:
                print("       traderequest: {}={}".format(tradereq_filed,traderequest_dict[tradereq_filed]))
    print("shutdown() and quit")
    mt5.shutdown()
    quit()
 
print("2. order_send done, ", result)
print("   opened position with POSITION_TICKET={}".format(result.order))
print("   sleep 2 seconds before closing position #{}".format(result.order))
time.sleep(2)
# create a close request
position_id=result.order
price=mt5.symbol_info_tick(symbol).bid
deviation=20
request={
    "action": mt5.TRADE_ACTION_DEAL,
    "symbol": symbol,
    "volume": lot,
    "type": mt5.ORDER_TYPE_SELL,
    "position": position_id,
    "price": price,
    "deviation": deviation,
    "magic": 234000,
    "comment": "python script close",
    "type_time": mt5.ORDER_TIME_GTC,
    "type_filling": mt5.ORDER_FILLING_RETURN,
}
# send a trading request
result=mt5.order_send(request)
# check the execution result
print("3. close position #{}: sell {} {} lots at {} with deviation={} points".format(position_id,symbol,lot,price,deviation));
if result.retcode != mt5.TRADE_RETCODE_DONE:
    print("4. order_send failed, retcode={}".format(result.retcode))
    print("   result",result)
else:
    print("4. position #{} closed, {}".format(position_id,result))
    # request the result as a dictionary and display it element by element
    result_dict=result._asdict()
    for field in result_dict.keys():
        print("   {}={}".format(field,result_dict[field]))
        # if this is a trading request structure, display it element by element as well
        if field=="request":
            traderequest_dict=result_dict[field]._asdict()
            for tradereq_filed in traderequest_dict:
                print("       traderequest: {}={}".format(tradereq_filed,traderequest_dict[tradereq_filed]))
 
# shut down connection to the MetaTrader 5 terminal
mt5.shutdown()

NameError: name 'mt5' is not defined

In [19]:
import MetaTrader5 as mt5
import time
import logging

class Trade_Executor:
    """
    Handles connection, data retrieval, and trade execution with MetaTrader 5.
    """
    
    def __init__(self, login=None, password=None, server=None, path=None):
        self.login = login
        self.password = password
        self.server = server
        self.path = path
        self.magic = 10000 # Unique ID for your bot orders

    def initialize_mt5(self):
        """Initializes the connection to the MT5 terminal."""
        logging.info("Attempting to initialize MetaTrader 5...")
        
        # Check if already initialized
        if mt5.terminal_info():
            logging.info("MT5 terminal already running and connected.")
            return True

        # Initialize with optional parameters
        if mt5.initialize(
            login=self.login, 
            password=self.password, 
            server=self.server, 
            path=self.path
        ):
            logging.info(f"✅ MT5 initialized successfully. Account: {mt5.account_info().login}")
            return True
        else:
            error_code = mt5.last_error()
            logging.error(f"❌ MT5 initialization failed. Error code: {error_code}")
            return False

In [22]:
executor =  Trade_Executor()
executor.initialize_mt5()


True